# Mandelbrot Set — Multi-Cell REPL

Demonstrates the **incremental REPL** style of xeus-lfortran.
Variables declared in one cell persist in the global symbol table and are
available in every subsequent cell — just like a Python/Julia REPL.

Because escape counts are computed once and stored, we can re-colour
the image in later cells without recomputing.

## BMP encoder

In [ ]:
module bmp_display
  use lfortran_display, only: display_data
  implicit none
contains
  subroutine display_image(w, h, rgba)
    integer, intent(in) :: w, h, rgba(4,w,h)
    integer :: rs, pad, fs, r, c, idx
    character(len=:), allocatable :: bmp
    rs = w*3; pad = mod(4-mod(rs,4),4); fs = 54+(rs+pad)*h
    allocate(character(len=fs) :: bmp); bmp = repeat(char(0),fs)
    bmp(1:2) = 'BM'
    call le32(bmp,3,fs); call le32(bmp,11,54)
    call le32(bmp,15,40); call le32(bmp,19,w); call le32(bmp,23,h)
    call le16(bmp,27,1); call le16(bmp,29,24)
    idx = 55
    do r = h, 1, -1
      do c = 1, w
        bmp(idx:idx)     = char(min(255,max(0,rgba(3,c,r))))
        bmp(idx+1:idx+1) = char(min(255,max(0,rgba(2,c,r))))
        bmp(idx+2:idx+2) = char(min(255,max(0,rgba(1,c,r))))
        idx = idx + 3
      end do
      idx = idx + pad
    end do
    call display_data('image/bmp', b64(bmp))
  end subroutine
  subroutine le32(s,o,v); character(len=*),intent(inout)::s; integer,intent(in)::o,v
    s(o:o)=char(iand(v,255)); s(o+1:o+1)=char(iand(ishft(v,-8),255))
    s(o+2:o+2)=char(iand(ishft(v,-16),255)); s(o+3:o+3)=char(iand(ishft(v,-24),255))
  end subroutine
  subroutine le16(s,o,v); character(len=*),intent(inout)::s; integer,intent(in)::o,v
    s(o:o)=char(iand(v,255)); s(o+1:o+1)=char(iand(ishft(v,-8),255))
  end subroutine
  function b64(data) result(out)
    character(len=*),intent(in)::data; character(len=:),allocatable::out
    character(len=64),parameter::alpha='ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789+/'
    integer::n,i,j,v
    n=len(data); allocate(character(len=((n+2)/3)*4)::out); j=1
    do i=1,n,3
      v=ishft(ichar(data(i:i)),16)
      if(i+1<=n) v=v+ishft(ichar(data(i+1:i+1)),8)
      if(i+2<=n) v=v+ichar(data(i+2:i+2))
      out(j:j)=alpha(ishft(v,-18)+1:ishft(v,-18)+1)
      out(j+1:j+1)=alpha(iand(ishft(v,-12),63)+1:iand(ishft(v,-12),63)+1)
      out(j+2:j+2)=merge(alpha(iand(ishft(v,-6),63)+1:iand(ishft(v,-6),63)+1),'=',i+1<=n)
      out(j+3:j+3)=merge(alpha(iand(v,63)+1:iand(v,63)+1),'=',i+2<=n)
      j=j+4
    end do
  end function
end module bmp_display

## Step 1 — Parameters

In [ ]:
integer, parameter :: Nx = 600, Ny = 450, n_max = 255, dp = kind(0.d0)
real(dp), parameter :: xcenter = -0.5_dp, ycenter = 0.0_dp, &
    width = 4.0_dp, height = 3.0_dp, &
    dx_di = width/Nx,  dy_dj = -height/Ny, &
    x_offset = xcenter - (Nx+1)*dx_di/2, &
    y_offset = ycenter - (Ny+1)*dy_dj/2

## Step 2 — Declare arrays

In [ ]:
integer :: image(Nx, Ny), image_color(4, Nx, Ny), palette(3, 4)
integer :: i, j, n, idx
real(dp) :: x, y, x_0, y_0, x_sqr, y_sqr

## Step 3 — Compute escape iteration count

In [ ]:
do j = 1, Ny
    y_0 = y_offset + dy_dj * j
    do i = 1, Nx
        x_0 = x_offset + dx_di * i
        x = 0; y = 0; n = 0
        do
            x_sqr = x**2; y_sqr = y**2
            if (x_sqr + y_sqr > 4 .or. n == n_max) then
                image(i,j) = 255 - n
                exit
            end if
            y = y_0 + 2*x*y
            x = x_0 + x_sqr - y_sqr
            n = n + 1
        end do
    end do
end do
print *, 'Computation done.'

## Step 4 — Map to RGBA colour palette

In [ ]:
palette(1,1) =   0; palette(2,1) = 135; palette(3,1) =  68
palette(1,2) =   0; palette(2,2) =  87; palette(3,2) = 231
palette(1,3) = 214; palette(2,3) =  45; palette(3,3) =  32
palette(1,4) = 255; palette(2,4) = 167; palette(3,4) =   0

do j = 1, Ny
    do i = 1, Nx
        idx = mod(image(i,j), 4) + 1
        image_color(1,i,j) = palette(1,idx)
        image_color(2,i,j) = palette(2,idx)
        image_color(3,i,j) = palette(3,idx)
        image_color(4,i,j) = 255
    end do
end do

## Step 5 — Display (colour)

In [ ]:
use bmp_display
call display_image(Nx, Ny, image_color)

## Re-render in greyscale

Because all variables persist, we can re-colour without re-computing —
just remap the stored `image` escape counts directly to greyscale.

In [ ]:
use bmp_display
do j = 1, Ny
    do i = 1, Nx
        n = image(i,j)
        image_color(1,i,j) = n
        image_color(2,i,j) = n
        image_color(3,i,j) = n
        image_color(4,i,j) = 255
    end do
end do
call display_image(Nx, Ny, image_color)